## Interview Success Prediction

Given *data about resumes*, let's try to predict whether a candidate will **pass the interview** based on their resume.

We will use three models to make our predictions and PCA for dimensionality reduction to make our predictions.

Data source: https://www.kaggle.com/datasets/vingkan/strategeion-resume-skills

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
data = pd.read_csv('archive/resumes_pilot.csv')
data

,Unnamed: 0,Interview,Adobe Illustrator,Adobe Photoshop,Agile Methodologies,Analytical Skills,Art,Art Direction,Auditing,AutoCAD,...,Windows,Windows Server,Wireless Technologies,WordPress,Workshops,Writing,Veteran,Female,URM,Disability
0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
1,1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1981,1981,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1982,1982,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1983,1983,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1984,1984,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1986 entries, 0 to 1985
Columns: 224 entries, Unnamed: 0 to Disability
dtypes: float64(223), int64(1)
memory usage: 3.4 MB


### Preprocessing

In [4]:
df = data.copy()

In [6]:
# Checking for missing values
df.isna().sum().sum()

np.int64(0)

In [7]:
# Drop the index column
df = df.drop('Unnamed: 0', axis=1)

In [8]:
df

,Interview,Adobe Illustrator,Adobe Photoshop,Agile Methodologies,Analytical Skills,Art,Art Direction,Auditing,AutoCAD,Automation,...,Windows,Windows Server,Wireless Technologies,WordPress,Workshops,Writing,Veteran,Female,URM,Disability
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1981,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1982,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1983,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1984,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0


In [9]:
# Split df into X and y
y = df['Interview'].copy()
X = df.drop('Interview', axis=1).copy()

In [10]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=1)

In [11]:
X_train

,Adobe Illustrator,Adobe Photoshop,Agile Methodologies,Analytical Skills,Art,Art Direction,Auditing,AutoCAD,Automation,Automotive,...,Windows,Windows Server,Wireless Technologies,WordPress,Workshops,Writing,Veteran,Female,URM,Disability
1707,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1796,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
481,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1685,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1256,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1791,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1096,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
1932,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0
235,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0


In [12]:
y_train.value_counts()

Interview
0.0    943
1.0    447
Name: count, dtype: int64

In [13]:
# Scale X
scaler = StandardScaler()
scaler.fit(X_train)

X_train = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

In [14]:
X_train

,Adobe Illustrator,Adobe Photoshop,Agile Methodologies,Analytical Skills,Art,Art Direction,Auditing,AutoCAD,Automation,Automotive,...,Windows,Windows Server,Wireless Technologies,WordPress,Workshops,Writing,Veteran,Female,URM,Disability
1707,-0.148522,-0.17867,-0.360668,-0.165368,-0.148522,-0.165368,-0.165368,-0.167650,-0.153506,-0.160718,...,-0.140745,-0.151033,-0.145972,-0.185003,-0.160718,-0.158347,-0.770738,-0.997126,-0.956348,-0.903877
1796,-0.148522,-0.17867,2.772634,-0.165368,-0.148522,-0.165368,-0.165368,-0.167650,-0.153506,-0.160718,...,-0.140745,-0.151033,-0.145972,-0.185003,-0.160718,-0.158347,1.297458,1.002882,1.045645,1.106345
481,-0.148522,-0.17867,-0.360668,-0.165368,-0.148522,-0.165368,-0.165368,-0.167650,-0.153506,-0.160718,...,-0.140745,-0.151033,-0.145972,-0.185003,-0.160718,-0.158347,1.297458,-0.997126,-0.956348,1.106345
1685,-0.148522,-0.17867,2.772634,-0.165368,-0.148522,-0.165368,-0.165368,-0.167650,-0.153506,-0.160718,...,-0.140745,-0.151033,-0.145972,-0.185003,-0.160718,-0.158347,-0.770738,-0.997126,-0.956348,-0.903877
1256,-0.148522,-0.17867,-0.360668,-0.165368,-0.148522,-0.165368,-0.165368,-0.167650,-0.153506,-0.160718,...,-0.140745,-0.151033,-0.145972,-0.185003,-0.160718,-0.158347,-0.770738,-0.997126,-0.956348,-0.903877
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1791,-0.148522,-0.17867,-0.360668,-0.165368,-0.148522,-0.165368,-0.165368,-0.167650,-0.153506,-0.160718,...,-0.140745,-0.151033,-0.145972,-0.185003,-0.160718,-0.158347,-0.770738,-0.997126,-0.956348,-0.903877
1096,-0.148522,-0.17867,-0.360668,-0.165368,-0.148522,-0.165368,-0.165368,-0.167650,-0.153506,-0.160718,...,-0.140745,-0.151033,-0.145972,-0.185003,-0.160718,-0.158347,1.297458,1.002882,1.045645,1.106345
1932,-0.148522,-0.17867,-0.360668,-0.165368,-0.148522,-0.165368,-0.165368,-0.167650,-0.153506,-0.160718,...,-0.140745,-0.151033,-0.145972,-0.185003,-0.160718,-0.158347,1.297458,1.002882,1.045645,1.106345
235,-0.148522,-0.17867,2.772634,-0.165368,-0.148522,-0.165368,-0.165368,5.964809,-0.153506,-0.160718,...,-0.140745,-0.151033,-0.145972,-0.185003,-0.160718,-0.158347,-0.770738,1.002882,1.045645,1.106345


### Training / Results

In [15]:
models = {
    'Logistic Regression': LogisticRegression(),
    '      Decision Tree': DecisionTreeClassifier(),
    '      Random Forest': RandomForestClassifier()
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name + " trained.")

Logistic Regression trained.
      Decision Tree trained.
      Random Forest trained.


In [16]:
for name, model in models.items():
    print(name + " Accuracy: {:.2f}%".format(model.score(X_test, y_test) * 100))

Logistic Regression Accuracy: 98.15%
      Decision Tree Accuracy: 87.42%
      Random Forest Accuracy: 92.79%


### Training / Results with Dimensionality Reduction

In [35]:
n_components = 3

pca = PCA(n_components = n_components)

pca.fit(X_train)

X_train_reduced = pd.DataFrame(pca.transform(X_train), index=X_train.index, columns=['PC' + str(i) for i in range(1, n_components + 1)])
X_test_reduced = pd.DataFrame(pca.transform(X_test), index=X_test.index, columns=['PC' + str(i) for i in range(1, n_components + 1)])

In [36]:
X_train_reduced

,PC1,PC2,PC3
1707,-1.736487,-0.437410,-0.454656
1796,2.964441,0.320901,-0.894002
481,1.823235,-1.806023,-0.490586
1685,-1.466734,-1.591823,-0.770624
1256,-0.184065,-1.728623,2.272956
...,...,...,...
1791,-1.816583,-0.862638,1.997459
1096,2.915615,0.352362,0.066201
1932,3.167263,0.514899,0.518571
235,0.346485,1.278118,-0.501213


In [37]:
pca.explained_variance_

array([4.21165636, 2.49185747, 1.86769427])

In [38]:
models = {
    'Logistic Regression': LogisticRegression(),
    '      Decision Tree': DecisionTreeClassifier(),
    '      Random Forest': RandomForestClassifier()
}

for name, model in models.items():
    model.fit(X_train_reduced, y_train)
    print(name + " trained.")

for name, model in models.items():
    print(name + " Accuracy: {:.2f}%".format(model.score(X_test_reduced, y_test) * 100))

Logistic Regression trained.
      Decision Tree trained.
      Random Forest trained.
Logistic Regression Accuracy: 98.15%
      Decision Tree Accuracy: 97.82%
      Random Forest Accuracy: 98.49%
